# Make FCTD for Brendan

In [ ]:
from nescafe23_fluctuations import tools
import numpy as np
from scipy import io
import pygenray as pr
import h5py
import xarray as xr
import cmps

In [ ]:
fctd = tools.compute_filtered_fctd_grid(z_smooth=None, r_smooth=None)

# shift front location
front_location = 35e3  # m

fctd = fctd.assign_coords({"range": (fctd.range.values + front_location)})

c = fctd["c"]

c.range.attrs['units'] = 'm'
c.depth.attrs['units'] = 'm'

c = c.interp({'range':np.arange(0,160000, 200)})

In [ ]:
mat_dict = {
    'c': c.values,
    # Save each coordinate as its own array, plus its attrs as coord_ATTRNAME fields
    **{f'{k}': np.array(v.values, dtype=object if v.values.dtype.kind in 'UO' else v.values.dtype)
       for k, v in c.coords.items()},
    **{f'{k}_{attr_name}': attr_val
       for k, v in c.coords.items()
       for attr_name, attr_val in v.attrs.items()},
    # Save DataArray attrs as individual fields
    **{f'{k}': val for k, val in c.attrs.items()},
    'name': c.name or '',
}

In [ ]:
io.savemat('/Users/john/Desktop/fctd_2D_field_nescafe23.mat', {'fctd':mat_dict})

## Run rays through result

In [ ]:
from nescafe23_fluctuations import tools
import numpy as np
from scipy import io
import scipy
import pygenray as pr
import h5py
from matplotlib import pyplot as plt
import cmps
import warnings
import xarray as xr

%config InlineBackend.figure_format = 'retina'

In [ ]:
def sub_fourier_synthesis(c_ref, x, freqs, phi_rec, pad_factor=0):
    """
    Compute Fourier synthesis time-domain signal using a complex envelope.

    Raises
    ------
    ValueError if inputs are inconsistent or improperly shaped.
    """

    x = np.asarray(x)
    freqs = np.asarray(freqs)
    phi_rec = np.asarray(phi_rec)

    nn = phi_rec.shape

    # Validate dimensionality
    if len(nn) not in (2, 3):
        raise ValueError("phi_rec must be 2D or 3D.")

    # Find matching axes
    ix = [i for i, n in enumerate(nn) if n == x.size]
    iw = [i for i, n in enumerate(nn) if n == freqs.size]

    if len(ix) != 1:
        raise ValueError("Could not uniquely match an axis of phi_rec to x.")
    if len(iw) != 1:
        raise ValueError("Could not uniquely match an axis of phi_rec to freqs.")

    ix = ix[0]
    iw = iw[0]

    # Reorder phi_rec so axis 0 = frequency, axis 1 = x
    if len(nn) == 3:
        iz = [i for i in range(3) if i not in (ix, iw)][0]
        pvec = [iw, ix, iz]
    else:
        pvec = [iw, ix]

    greens = np.transpose(phi_rec, axes=pvec)

    # Validate frequency spacing
    if freqs.size < 2:
        raise ValueError("freqs must contain at least two values.")

    df = abs(freqs[1] - freqs[0])
    if not np.allclose(np.diff(freqs), df):
        raise ValueError("freqs must be uniformly spaced.")

    N = freqs.size

    # Time shift
    tt = x / c_ref

    # Padding
    pad_factor = int(round(pad_factor))
    if pad_factor < 0:
        raise ValueError("pad_factor must be >= 0.")

    # Hann window
    window = np.hanning(N)

    # Phase term
    phase = np.exp(-1j * 2 * np.pi * freqs[:, np.newaxis] * tt[np.newaxis, ...])

    # Apply window and phase
    reshape_window = (N,) + (1,) * (greens.ndim - 1)
    dsignal = window.reshape(reshape_window) * greens * phase

    # FFT size
    nfft = 2 ** (int(np.ceil(np.log2(N))) + pad_factor)

    # FFT
    ptime = np.fft.fft(dsignal, n=nfft, axis=0) / np.sum(window)

    # Time axis
    M = ptime.shape[0]
    dt = 1.0 / (M * df)
    tp = dt * np.arange(M)

    # Restore original axis order
    inv_pvec = np.argsort(pvec)
    ptime = np.transpose(ptime, axes=inv_pvec)

    return ptime, tt, tp

import numpy as np


def fourier_synthesis_along_axis(c_ref, freqs, x, phi_rec, axis=0, pad_factor=0):
    """
    Convert the frequency axis of phi_rec into a time axis using Fourier synthesis.

    Parameters
    ----------
    c_ref : float
        Reference sound speed.
    freqs : array_like
        1D frequency array, assumed uniformly spaced.
    x : float
        Scalar range associated with phi_rec.
    phi_rec : array_like
        Complex Green's function or frequency-domain field.
    axis : int, optional
        Axis of phi_rec corresponding to frequency. Default is 0.
    pad_factor : int, optional
        Extra zero-padding factor for the FFT:
            nfft = 2**(ceil(log2(N)) + pad_factor)
        Default is 0.

    Returns
    -------
    ptime : ndarray
        Time-domain complex envelope, with the same shape as phi_rec
        except along `axis`, which becomes the time dimension.
    tt : float
        Reference travel time x / c_ref.
    tp : ndarray
        Time axis corresponding to the transformed dimension.

    Raises
    ------
    ValueError
        If inputs are invalid.
    """

    freqs = np.asarray(freqs)
    phi_rec = np.asarray(phi_rec)

    if freqs.ndim != 1:
        raise ValueError("freqs must be a 1D array.")

    if freqs.size < 2:
        raise ValueError("freqs must contain at least two values.")

    if not np.isscalar(x):
        raise ValueError("x must be a scalar.")

    if not np.isscalar(c_ref):
        raise ValueError("c_ref must be a scalar.")

    axis = int(axis)
    if axis < -phi_rec.ndim or axis >= phi_rec.ndim:
        raise ValueError("axis is out of bounds for phi_rec.")

    axis = axis % phi_rec.ndim

    if phi_rec.shape[axis] != freqs.size:
        raise ValueError(
            f"Length of freqs ({freqs.size}) must match phi_rec.shape[axis] "
            f"({phi_rec.shape[axis]})."
        )

    df = abs(freqs[1] - freqs[0])
    if not np.allclose(np.diff(freqs), np.diff(freqs)[0]):
        raise ValueError("freqs must be uniformly spaced.")

    pad_factor = int(round(pad_factor))
    if pad_factor < 0:
        raise ValueError("pad_factor must be >= 0.")

    N = freqs.size
    tt = x / c_ref

    window = np.hanning(N)

    # Move frequency axis to front
    greens = np.moveaxis(phi_rec, axis, 0)

    # Apply window and reference travel-time phase shift
    reshape_window = (N,) + (1,) * (greens.ndim - 1)
    phase = np.exp(-1j * 2 * np.pi * freqs * tt).reshape(reshape_window)
    dsignal = greens * window.reshape(reshape_window) * phase

    # FFT along frequency axis
    nfft = 2 ** (int(np.ceil(np.log2(N))) + pad_factor)
    ptime = np.fft.fft(dsignal, n=nfft, axis=0) / np.sum(window)

    # Time axis
    M = ptime.shape[0]
    dt = 1.0 / (M * df)
    tp = dt * np.arange(M)

    # Move time axis back to original axis position
    ptime = np.moveaxis(ptime, 0, axis)

    return ptime, tt, tp

In [ ]:
fn = '/Volumes/OADL/PersonalStorage/Decourcy/NESCAFE-Models/SIOE-to-SION-PE-(FCTD03)/ModelResult_2DBBPE_IO.mat'

pe_results = h5py.File(fn)

In [ ]:
PEdata = pe_results['PEdata']['VARS']
PEenv = pe_results['PEenv']['VARS']

pf = PEdata['afield'][:].view(np.complex128)

In [ ]:
c_np = PEenv['cfield'][:]
c = xr.DataArray(c_np, dims=['depth', 'range'], coords={'depth':PEenv['zc'][:].flatten(), 'range':PEenv['xc'][:].flatten()})

In [ ]:
ptime, tt, tp = fourier_synthesis_along_axis(1500, PEdata['freqs'][:].flatten(), 153e3, pf, axis=0)

ptime = xr.DataArray(ptime, dims=['time', 'depth'], coords={'time':tp + tt, 'depth':PEdata['za'][:].flatten()})

In [ ]:
fig = plt.figure(figsize=(10,3))

(20*np.log10(np.abs(ptime))).plot(x='time', vmin=-130, vmax=-90, cmap=cmps.mako_r_white())

#plt.xlim([0,15])
plt.ylim([5000,0])

### run rays

In [ ]:
source_depth = float(PEdata['zs'][:])
receiver_range = float(PEdata['xr'][:])
bathy = xr.DataArray(
    np.linspace(5000, 5000, 100),  # flat bottom
    dims=["range"],
    coords={"range": np.linspace(0, 200e3, 100)},
)
envi = pr.OceanEnvironment2D(c, bathy)
launch_angles = np.linspace(-30,30,10000)

In [ ]:
rays = pr.shoot_rays(source_depth, 0, launch_angles, receiver_range, 3, envi)

In [ ]:
fig = plt.figure(figsize=(10,4))

(20*np.log10(np.abs(ptime))).plot(x='time', vmin=-130, vmax=-90, cmap=cmps.timefront(), cbar_kwargs={'label':'dB rel $\\mu Pa$'}, rasterized=True)

plt.scatter(rays.ts[:,-1], -rays.zs[:,-1], c='k', lw=0, s=3)

plt.ylim([5100,0])
plt.xlim([102,104])

plt.xlabel('time [s]')
plt.ylabel('depth [m]')

fig.savefig('figures/PE_rays_ducted.pdf', dpi=300)

In [ ]:
fig = plt.figure(figsize=(10,4))

(20*np.log10(np.abs(ptime))).plot(x='time', vmin=-130, vmax=-90, cmap=cmps.timefront(), cbar_kwargs={'label':'dB rel $\\mu Pa$'}, rasterized=True)

plt.scatter(rays.ts[:,-1], -rays.zs[:,-1], c='k', lw=0, s=3)

plt.ylim([5100,0])
plt.xlim([102,115])

plt.xlabel('time [s]')
plt.ylabel('depth [m]')

fig.savefig('figures/PE_rays_full.pdf', dpi=300)